# Kiểm tra từng bước truy xuất lai
Chạy thủ công sau khi dựng kho. Nhà cung cấp dùng theo cấu hình của `build_container()`.
Hiển thị mã, điểm, đường dẫn mục và 120 ký tự đầu của nguồn ở từng bước.
Không sử dụng văn bản ca bệnh hoặc hiển thị khoá.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "app").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from app.container import build_container
from app.retrieval.query_builder import QueryBuilder
from app.retrieval.setup import build_retriever

container = build_container()
retriever = build_retriever(container)
builder = QueryBuilder(container.settings.retrieval)

In [ ]:
from app.core.types import ScoredChunk
from app.retrieval.fusion import rrf

query = builder.for_finding("meropenem", "meropenem", "meropenem", 6, None)

def show(title: str, results: list[ScoredChunk]) -> None:
    print("\n" + title)
    for result in results[:5]:
        chunk = result.chunk
        print(chunk.doan_trich_id, result.diem, chunk.duong_dan_muc,
              chunk.noi_dung_goc[:120])

keyword_lists = [
    retriever.keyword_index.search(text, query.k_query, query.bo_loc)
    for text in query.van_ban
]
vector_lists = [
    retriever.vector_store.search(retriever.embedder.embed_query(text),
                                 query.k_query, query.bo_loc)
    for text in query.van_ban
]
for text, results in zip(query.van_ban, keyword_lists, strict=True):
    show("Từ khoá: " + text, results)
for text, results in zip(query.van_ban, vector_lists, strict=True):
    show("Véc-tơ: " + text, results)

In [ ]:
chunks = {}
rankings = []
for keyword_results, vector_results in zip(keyword_lists, vector_lists, strict=True):
    for results in (keyword_results, vector_results):
        rankings.append([result.chunk.doan_trich_id for result in results])
        chunks.update({result.chunk.doan_trich_id: result.chunk for result in results})
fused = rrf(rankings, retriever.rrf_k)[:query.k_query]
fused_results = [ScoredChunk(chunk=chunks[chunk_id], diem=score, nguon_diem="rrf")
                 for chunk_id, score in fused]
show("Sau RRF", fused_results)

candidates = [result.chunk for result in fused_results]
ranked = retriever.reranker.rerank(query.van_ban[0],
                                  [chunk.noi_dung_nhung for chunk in candidates],
                                  query.k_rerank)
reranked_results = [ScoredChunk(chunk=candidates[index], diem=score,
                               nguon_diem="xep_hang_lai")
                   for index, score in ranked]
show("Sau xếp hạng lại", reranked_results)